# 5 · Property enrichment  (run once)

Builds the schema.org **property** retrieval vocabulary that the relation-matching notebook
(`7_property_extension.ipynb`) depends on — the property-side twin of `1_schema_enrichment.ipynb`
(which does the same for *classes*). **One-time**: re-run only to refresh the definitions.

Steps:
1. **Parse** `schemaorg-current-https.jsonld` into ~1521 properties, each with its `domain → range`
   signature, `subPropertyOf` parent(s), and `inverseOf`.
2. **Enrich** every property comment with Gemini (batched, cached, interrupt-safe).
3. **Build** `schema_properties_enriched.json` — the enriched document pool (Examples / "Distinct
   from" clauses stripped from the embedded text, kept in `enriched_comment`).

Outputs (all under `schema_org_files/`): `schema_properties.json`, `enriched_prop_defs_cache.json`,
`schema_properties_enriched.json`. Mirrors nb1 field-for-field so nb7 can treat the two pools
(classes for types, properties for relations) the same way.

In [ ]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [ ]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

SCHEMA_DIR = ROOT / "schema_org_files"     # shared schema.org assets
GEMINI_MODEL = "gemini-2.5-flash"         # authors the enriched definitions

# Enrichment knobs
ENRICH_BATCH = 100                        # schema properties per Gemini call
MAX_PROPS    = None                       # set an int for a quick dry run; None = all ~1521 properties
GEMINI_PROP_DEFS_CACHE = SCHEMA_DIR / "enriched_prop_defs_cache.json"   # label -> enriched def (resumable)
print("gemini:", GEMINI_MODEL, "| schema dir:", SCHEMA_DIR)

In [ ]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)

# Token + latency accounting lives in notebooks/nb_parallel.py so every notebook reports the same
# numbers (prompt / output / THINKING tokens, per-call latency) from one ledger.
import sys as _sys, pathlib as _pl, importlib as _il
for _p in (_pl.Path.cwd(), *_pl.Path.cwd().parents):
    if (_p / "nb_parallel.py").exists():
        _sys.path.insert(0, str(_p)); break
import nb_parallel; _il.reload(nb_parallel); nb_parallel.install()

last_latency_s = None
LAST_USAGE = None

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")


# A 429 is retryable when it means "too fast" and pointless when it means "out of money": backing
# off five times against a depleted balance just burns three minutes before failing anyway.
_FATAL_WORDS = ("credits are depleted", "billing", "quota exceeded for quota metric",
                "exceeded your current quota", "insufficient")


def _gemini_retryable(exc) -> bool:
    """A transient server-side failure, as opposed to one that repeating cannot fix."""
    msg = str(exc).lower()
    if any(w in msg for w in _FATAL_WORDS):
        return False
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    return any(w.lower() in msg for w in _RETRY_WORDS)


def _gemini_call_with_retry(fn, attempts=5, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    ))
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return json.loads(resp.text)

## 1. Parse schema.org into properties with domain → range signatures

schema.org uses `schema:domainIncludes` / `schema:rangeIncludes` (a property may apply to several
domain types and accept several range types), `rdfs:subPropertyOf` for the property hierarchy, and
`schema:inverseOf` for inverse pairs. We keep all four — `domain → range` pins the intended sense the
same way the ancestor `path` does for classes.

In [ ]:
import re

def plain(value) -> str:
    if isinstance(value, dict):
        value = value.get("@value", "")
    value = re.sub(r"<[^>]+>", " ", value or "")
    return re.sub(r"\s+", " ", value).strip()

def as_list(x):
    if x is None: return []
    return x if isinstance(x, list) else [x]

def short(ref):
    """{'@id': 'schema:Person'} (or 'schema:Person') -> 'Person'."""
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]

raw = json.loads((SCHEMA_DIR / "schemaorg-current-https.jsonld").read_text())
props = {}
for node in raw["@graph"]:
    if "rdf:Property" not in as_list(node.get("@type")):
        continue
    p_id = node["@id"]
    if not p_id.startswith("schema:"):
        continue
    props[p_id] = {
        "id": p_id,
        "label": plain(node.get("rdfs:label")) or p_id.split(":")[-1],
        "comment": plain(node.get("rdfs:comment")),
        "domain": sorted({short(d) for d in as_list(node.get("schema:domainIncludes"))}),
        "range":  sorted({short(r) for r in as_list(node.get("schema:rangeIncludes"))}),
        "parents": [short(p) for p in as_list(node.get("rdfs:subPropertyOf"))
                    if isinstance(p, (str, dict))],
        "inverse_of": short(node.get("schema:inverseOf")) if node.get("schema:inverseOf") else None,
    }

def signature(p):
    """'Domain/... → Range/...' — the property analogue of a class ancestor path, for HUMAN display
    and the LLM judge prompt (which reasons about domain/range + direction). ANY when open. This is
    kept in the `signature` FIELD but is deliberately NOT put in `embedding_text` — see below."""
    d = "/".join(p["domain"]) if p["domain"] else "ANY"
    r = "/".join(p["range"])  if p["range"]  else "ANY"
    return f"{d} → {r}"

prop_list = list(props.values())
print(f"{len(prop_list)} schema.org properties parsed")

In [ ]:
# Persist the parsed properties: the baseline document pool (terse schema.org comment). The
# embedding_text is "label: comment" — the SAME shape as nb7's relation-phrase QUERY ("phrase:
# definition Also called synonyms."), so query and document sit in one style for the bi-encoder.
# The domain -> range signature is intentionally NOT embedded (the query has no counterpart for it,
# and the enriched definition already states the direction in prose); it lives in the `signature`
# field and is shown to the LLM judge instead. Same as schema_classes.json vs the enriched pool.
schema_properties = [
    {"id": p["id"], "label": p["label"], "comment": p["comment"],
     "domain": p["domain"], "range": p["range"], "sub_property_of": p["parents"],
     "inverse_of": p["inverse_of"], "signature": signature(p),
     "embedding_text": f'{p["label"]}: {p["comment"]}'}
    for p in prop_list
]
(SCHEMA_DIR / "schema_properties.json").write_text(json.dumps(schema_properties, indent=2, ensure_ascii=False))
print(f"wrote {len(schema_properties)} properties -> schema_properties.json")

## 2. Enrich schema property comments with Gemini

Batched and cached to `schema_org_files/enriched_prop_defs_cache.json`. Interrupt-safe: re-running
skips properties already in the cache, so you only pay for what's missing. Set `MAX_PROPS` in the
config cell for a quick partial run.

In [ ]:
ENRICH_PROP_SYSTEM_PROMPT = """You are building a retrieval vocabulary for the schema.org property (relation) system. You are given a batch of schema.org properties, each with its domain -> range signature (the subject types it applies to and the object types it accepts, e.g. "Organization/Person -> Person") and its original schema.org comment. A property expresses a relation that holds from a subject entity to an object entity. For each property, write one enriched definition that will later be embedded and matched against free-form relation phrases extracted from text.

Write each definition as a few compact sentences in this exact order and style:
1. A precise one-sentence gloss describing the relation as a neutral association between the two kinds of entity it connects. Do NOT fix which entity is the subject and which is the object, do NOT use the words "subject"/"object", and do NOT phrase the direction (e.g. avoid "X has a Y" or "the Y of X"). Phrase it so that both this property and its inverse would read as relevant — retrieval matches the relation concept, and direction is resolved separately from the domain -> range signature, not from this text. Name the two participant roles and the core meaning, e.g. "An authorship relation between a creative work and the person or organization that created it." or "An employment relation between a person and an organization." Use the domain -> range signature only to pin down which relation this is (its sense), NEVER to encode a direction in the wording.
2. "Also called <2-5 synonyms or alternative phrasings of the relation>." (verb phrases are fine, e.g. for "author": "written by", "created by", "authored by").
3. "Examples: <3-6 concrete subject-relation-object instances>."
4. ONLY if the property is easily confused with a sibling property, add: "Distinct from <sibling>, which is <one short clause>." Otherwise omit this sentence.

Keep every definition short and information-dense: long generic prose dilutes a mean-pooled embedding. Do not restate the label verbatim as the whole gloss. Stay faithful to schema.org's intended meaning and to the relation's direction.

Return ONLY valid JSON in this schema, covering every label in the batch exactly once:
{ "definitions": { "<label>": "<enriched definition>", "...": "..." } }"""

defs = json.loads(GEMINI_PROP_DEFS_CACHE.read_text()) if GEMINI_PROP_DEFS_CACHE.exists() else {}

pool = prop_list if MAX_PROPS is None else prop_list[:MAX_PROPS]
todo = [p for p in pool if p["label"] not in defs]
print(f"{len(defs)} cached | {len(todo)} to enrich | batch={ENRICH_BATCH}")

nb_parallel.reset_usage()   # the report below covers only this run
MAX_RETRIES = 5          # attempts per batch before giving up (transient errors: rate limits, timeouts)
t_start = time.perf_counter()
for i in range(0, len(todo), ENRICH_BATCH):
    batch = todo[i:i + ENRICH_BATCH]
    payload = [{"label": p["label"], "signature": signature(p), "comment": p["comment"]} for p in batch]
    bnum = i // ENRICH_BATCH

    # Retry with exponential backoff; do NOT skip. If a batch can't be enriched after MAX_RETRIES,
    # persist what we have and stop hard so no properties are silently dropped (the cache lets you
    # resume by simply re-running this cell).
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            out = gemini_json(ENRICH_PROP_SYSTEM_PROMPT, json.dumps(payload, ensure_ascii=False))
            got = out.get("definitions", {})
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                GEMINI_PROP_DEFS_CACHE.write_text(json.dumps(defs, indent=2, ensure_ascii=False))
                raise RuntimeError(
                    f"batch {bnum} failed after {MAX_RETRIES} attempts: {e}. "
                    f"{len(defs)} defs cached — re-run this cell to resume from here."
                ) from e
            wait = min(2 ** attempt, 30)
            print(f"  batch {bnum} attempt {attempt}/{MAX_RETRIES} failed ({e}); retrying in {wait}s ...")
            time.sleep(wait)

    # keep only non-empty defs for labels we actually asked about
    missing = []
    for p in batch:
        d = got.get(p["label"])
        if d and d.strip():
            defs[p["label"]] = d.strip()
        else:
            missing.append(p["label"])
    if missing:
        print(f"  batch {bnum}: model omitted {len(missing)} label(s), will retry next run: {missing[:8]}")
    GEMINI_PROP_DEFS_CACHE.write_text(json.dumps(defs, indent=2, ensure_ascii=False))
    done = min(i + ENRICH_BATCH, len(todo))
    print(f"  {len(defs)} total enriched  (+{done}/{len(todo)} this run, {last_latency_s:.1f}s/call)")

remaining = [p["label"] for p in pool if p["label"] not in defs]
print(f"done in {time.perf_counter()-t_start:.0f}s | {len(defs)}/{len(pool)} labels have a Gemini definition")
if remaining:
    print(f"{len(remaining)} still missing (model omitted them from its JSON) — re-run this cell to fill them: {remaining[:12]}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
nb_parallel.usage_report("nb5 · schema property enrichment", wall_seconds=time.perf_counter() - t_start,
                         write=SCHEMA_DIR / "token_usage_nb5_property_enrichment.json")

## 3. Build the enriched document pool

In [ ]:
# Build the enriched document pool. Strip the "Examples: ..." sentence and contrastive
# "Distinct from ..." clauses from the embedded text (they hurt a bi-encoder), but keep the
# full text in enriched_comment. Identical treatment to nb1's class pool.
# Strip the contrastive "Distinct from ..." clause first, then everything from "Examples:" to the end.
# "Examples:" is the last templated element (order: gloss, "Also called", "Examples", optional
# "Distinct from" which we remove first), so anchoring the example strip to end-of-string (DOTALL) is
# robust — the earlier non-greedy ".*?\." version broke on internal periods like "J.K. Rowling".
_CONTRAST_RE = re.compile(r"\s*(?:Distinct from|Not to be confused with)[^.]*\.", re.IGNORECASE)
_EXAMPLE_RE  = re.compile(r"\s*Examples?:.*\Z", re.IGNORECASE | re.DOTALL)

def embedding_comment(text):
    text = _CONTRAST_RE.sub("", text)
    text = _EXAMPLE_RE.sub("", text)
    return text.strip()

enriched_props = []
for p in prop_list:
    ec = defs.get(p["label"])
    is_en = ec is not None
    comment_for_embed = embedding_comment(ec) if is_en else p["comment"]
    enriched_props.append({
        "id": p["id"], "label": p["label"], "comment": p["comment"],
        "domain": p["domain"], "range": p["range"], "sub_property_of": p["parents"],
        "inverse_of": p["inverse_of"], "signature": signature(p),
        "enriched_comment": ec or "", "enriched": is_en,
        "embedding_text": f'{p["label"]}: {comment_for_embed}',
    })

out_pool = SCHEMA_DIR / "schema_properties_enriched.json"
out_pool.write_text(json.dumps(enriched_props, indent=2, ensure_ascii=False))
print(f"{sum(p['enriched'] for p in enriched_props)}/{len(enriched_props)} enriched -> {out_pool}")
# spot check
for lbl in ["founder", "author", "birthPlace", "memberOf"]:
    m = next((p["embedding_text"] for p in enriched_props if p["label"] == lbl), None)
    if m: print("  ", m)